# Compile YOLO11n -> Hailo-8 HEF (lathe-inspector)

Converts `best.onnx` (4 defect classes: bent, color, flip, scratch) into a
quantized `.hef` that runs on the Hailo-8 AI HAT+.

**Before running:** your Google Drive must contain a folder `hailo/` with:
1. `hailo_dataflow_compiler-*-py3-none-linux_x86_64.whl` (from the Hailo Developer Zone)
2. `best.onnx`
3. `hailo_calib.zip` (unlabeled nut images for quantization calibration)

Run cells top to bottom. The compile cell takes 20-60 min on the free CPU runtime.
When everything finishes, `yolov11n.hef` appears in your Drive's `hailo/` folder.

In [ ]:
# Cell 1: mount Google Drive (grants this VM access to your hailo/ folder)
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/hailo/

In [ ]:
# Cell 2: create an isolated Python environment.
# DFC v3.34 supports Python 3.10/3.11/3.12, so Colab's default Python is
# fine -- but the compiler drags in its own TensorFlow and friends, so we
# keep it in a venv instead of polluting Colab's preinstalled packages.
!sudo apt-get update -qq && sudo apt-get install -y -qq python3-venv
!python3 -m venv /content/hailo_env
!/content/hailo_env/bin/pip install --upgrade pip -q
!/content/hailo_env/bin/python --version

In [ ]:
# Cell 3: install the Dataflow Compiler (the big wheel from your Drive)
# and the Hailo Model Zoo (provides the `hailomz` CLI + the yolov11n
# network recipe so we don't hand-write compiler configs).
!/content/hailo_env/bin/pip install /content/drive/MyDrive/hailo/hailo_dataflow_compiler-*.whl -q
!git clone --depth 1 https://github.com/hailo-ai/hailo_model_zoo.git /content/hailo_model_zoo
!/content/hailo_env/bin/pip install -e /content/hailo_model_zoo -q
!/content/hailo_env/bin/hailomz --version

In [ ]:
# Cell 4: stage the model + calibration images on the VM's local disk
!mkdir -p /content/work
!cp /content/drive/MyDrive/hailo/best.onnx /content/work/
!unzip -q -o /content/drive/MyDrive/hailo/hailo_calib.zip -d /content/work/
!ls /content/work/hailo_calib | wc -l   # how many calibration images

In [ ]:
# Cell 5: COMPILE (the long one -- 20-60 min, watch for 'HEF file written').
# What happens inside: the compiler parses the ONNX graph, quantizes
# float32 weights to int8 using your calibration images to pick the value
# ranges, then maps the whole network onto Hailo-8's dataflow cores.
#   --classes 4  : our model has 4 defect classes, not COCO's 80
#   --hw-arch hailo8l : your chip is the Hailo-8L (13 TOPS) -- confirmed via hailortcli
%cd /content/work
!/content/hailo_env/bin/hailomz compile yolov11n \
  --ckpt /content/work/best.onnx \
  --calib-path /content/work/hailo_calib \
  --hw-arch hailo8l \
  --classes 4

In [ ]:
# Cell 6: find the .hef and save it back to Drive
!find /content/work -name '*.hef'
!cp $(find /content/work -name '*.hef' | head -1) /content/drive/MyDrive/hailo/
!ls -la /content/drive/MyDrive/hailo/

## Done
Download `yolov11n.hef` from your Drive's `hailo/` folder (or straight from the
file browser on the left). Next step: copy it to the Pi and verify with HailoRT.

**If Cell 5 fails**, the usual suspects:
- *version mismatch between model zoo and compiler*: the error names the expected
  version -- re-clone the model zoo at the matching tag, e.g.
  `git clone -b v2.14 https://github.com/hailo-ai/hailo_model_zoo.git`
- *out of memory*: rerun with `--optimization-level 0` added to the compile command
- *unsupported ONNX op*: re-export on the Mac with a different `opset` (try 13 or 11)